# Chapter 3: Implicit Differentiation Through a Fit

## The Complete Pipeline

1. Draw samples from N(mu, sigma=1) where mu is trainable
2. Square the samples: y = x^2
3. Fit a Voigt PDF to the y-values using maximum likelihood -> get gamma_fit
4. Loss = (gamma_fit - target)^2
5. Optimize mu via gradient descent

**The problem:** Step 3 (the fit) is an iterative optimization (L-BFGS) that blocks gradients.

**The solution:** Implicit differentiation -- we differentiate *around* the fit using the optimality condition.

## The Pseudo-Voigt PDF

The **pseudo-Voigt** is a weighted sum of a Gaussian and a Lorentzian:

$V(x) = \eta \cdot G(x) + (1-\eta) \cdot L(x)$

Fully differentiable in PyTorch.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

def pseudo_voigt_pdf(x, log_gamma, sigma_g, center, eta):
    """Pseudo-Voigt PDF. gamma = exp(log_gamma) to keep it positive."""
    gamma = torch.exp(log_gamma)
    gauss = torch.exp(-0.5 * ((x - center) / sigma_g) ** 2) / (sigma_g * torch.sqrt(torch.tensor(2 * torch.pi)))
    lorentz = (gamma / torch.pi) / ((x - center) ** 2 + gamma ** 2)
    return eta * gauss + (1 - eta) * lorentz

n_samples = 100
print('Ready')

## Step 1: Sample from Gaussian, Square, Fit Voigt

We draw samples from N(mu, 1), square them, and fit a Voigt PDF using maximum likelihood.

In [ ]:
# Generate data: 100 samples from N(mu=5, 1), then square
mu_true = 5.0
x_samp = torch.randn(n_samples) + mu_true
y_data = x_samp ** 2

print(f'y values: mean = {y_data.mean().item():.2f}, std = {y_data.std().item():.2f}')

# Fit Voigt using log_gamma (keeps gamma positive naturally)
log_gamma = torch.tensor([2.0], requires_grad=True)
center_fit = torch.tensor([30.0], requires_grad=True)

opt = torch.optim.LBFGS([log_gamma, center_fit], max_iter=200, lr=0.5)

def nll_loss():
    opt.zero_grad()
    pdf = pseudo_voigt_pdf(y_data.detach(), log_gamma, torch.tensor([5.0]), center_fit, torch.tensor([0.5]))
    loss = -torch.sum(torch.log(pdf + 1e-10))
    loss.backward()
    return loss

opt.step(nll_loss)
print(f'Fitted: gamma = {torch.exp(log_gamma).item():.2f}, center = {center_fit.item():.2f}')

# Visualize
x_grid = torch.linspace(0, 80, 500)
pdf_vals = pseudo_voigt_pdf(x_grid, log_gamma.detach(), torch.tensor([5.0]), center_fit.detach(), torch.tensor([0.5]))

plt.hist(y_data.numpy(), bins=15, density=True, alpha=0.5, label='Data')
plt.plot(x_grid.numpy(), pdf_vals.detach().numpy(), '-', linewidth=2, label=f'Voigt fit')
plt.xlabel('y = x^2'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 2: Implicit Differentiation -- One Step

At the optimal fit params, the inner loss gradient is zero. When we change `mu`, the data changes and the optimal params shift. We compute how:

$\frac{d\theta^*}{d\mu} = -\mathbf{H}^{-1} \cdot \frac{\partial^2 L}{\partial \theta \partial \mu}$

In [ ]:
mu = torch.tensor([5.0], requires_grad=True)
torch.manual_seed(42)
y_data_grad = (torch.randn(n_samples) + mu) ** 2

lg = torch.tensor([2.0], requires_grad=True)
cf = torch.tensor([30.0], requires_grad=True)
opt = torch.optim.LBFGS([lg, cf], max_iter=200, lr=0.5)
def closure():
    opt.zero_grad()
    pdf = pseudo_voigt_pdf(y_data_grad.detach(), lg, torch.tensor([5.0]), cf, torch.tensor([0.5]))
    loss = -torch.sum(torch.log(pdf + 1e-10))
    loss.backward()
    return loss
opt.step(closure)
print(f'Fit: gamma = {torch.exp(lg).item():.4f}')

# --- Implicit differentiation ---
lg_opt = lg.detach().clone().requires_grad_(True)
cf_opt = cf.detach().clone().requires_grad_(True)
pdf = pseudo_voigt_pdf(y_data_grad, lg_opt, torch.tensor([5.0]), cf_opt, torch.tensor([0.5]))
loss_inner = -torch.sum(torch.log(pdf + 1e-10))

grad_lg, grad_cf = torch.autograd.grad(loss_inner, [lg_opt, cf_opt], create_graph=True)
print(f'Gradient at optimum: dL/dlog_gamma = {grad_lg.item():.4f}, dL/dcenter = {grad_cf.item():.4f}')

# Hessian
H_ll = torch.autograd.grad(grad_lg, lg_opt, retain_graph=True)[0]
H_lc = torch.autograd.grad(grad_lg, cf_opt, retain_graph=True)[0]
H_cl = torch.autograd.grad(grad_cf, lg_opt, retain_graph=True)[0]
H_cc = torch.autograd.grad(grad_cf, cf_opt, retain_graph=True)[0]
H = torch.tensor([[H_ll.item(), H_lc.item()], [H_cl.item(), H_cc.item()]])
print(f'Hessian:\n{H}')

# Mixed derivatives
mixed_lg = torch.autograd.grad(grad_lg, mu, retain_graph=True)[0]
mixed_cf = torch.autograd.grad(grad_cf, mu, retain_graph=True)[0]

dtheta_dmu = -torch.linalg.solve(H, torch.tensor([[mixed_lg.item()], [mixed_cf.item()]]))
dgamma_dmu = dtheta_dmu[0,0].item() * torch.exp(lg_opt).item()
print(f'dgamma/dmu = {dgamma_dmu:.4f}')

## Step 3: Full Optimization Loop

Now we use the implicit gradient to actually optimize `mu` so that `gamma_fit` reaches a target value.

In [ ]:
target_gamma = 15.0
mu = torch.tensor([3.0])
lr = 0.08
n_steps = 30

history = []

for step in range(n_steps):
    torch.manual_seed(42)
    y_data_step = (torch.randn(n_samples) + mu) ** 2

    lg = torch.tensor([2.0], requires_grad=True)
    cf = torch.tensor([30.0], requires_grad=True)
    opt_inner = torch.optim.LBFGS([lg, cf], max_iter=200, lr=0.5)
    def closure():
        opt_inner.zero_grad()
        pdf = pseudo_voigt_pdf(y_data_step.detach(), lg, torch.tensor([5.0]), cf, torch.tensor([0.5]))
        loss = -torch.sum(torch.log(pdf + 1e-10))
        loss.backward()
        return loss
    opt_inner.step(closure)
    gamma_fitted = torch.exp(lg).detach().item()

    # Implicit diff
    mu_grad = mu.clone().requires_grad_(True)
    y_grad = (torch.randn(n_samples) + mu_grad) ** 2
    lg_opt = lg.detach().clone().requires_grad_(True)
    cf_opt = cf.detach().clone().requires_grad_(True)
    pdf = pseudo_voigt_pdf(y_grad, lg_opt, torch.tensor([5.0]), cf_opt, torch.tensor([0.5]))
    loss_inner = -torch.sum(torch.log(pdf + 1e-10))
    grad_lg, grad_cf = torch.autograd.grad(loss_inner, [lg_opt, cf_opt], create_graph=True)

    H = torch.zeros(2, 2)
    for idx, (g, p) in enumerate([(grad_lg, lg_opt), (grad_cf, cf_opt)]):
        for jdx, p2 in enumerate([lg_opt, cf_opt]):
            H[idx, jdx] = torch.autograd.grad(g, p2, retain_graph=True)[0].item()
    H += torch.eye(2) * 1e-3

    mixed = torch.zeros(2, 1)
    mixed[0] = torch.autograd.grad(grad_lg, mu_grad, retain_graph=True)[0].item()
    mixed[1] = torch.autograd.grad(grad_cf, mu_grad, retain_graph=True)[0].item()

    dtheta_dmu = -torch.linalg.solve(H, mixed)
    dgamma_dmu = dtheta_dmu[0,0].item() * torch.exp(lg_opt).item()

    grad_mu = 2 * (gamma_fitted - target_gamma) * dgamma_dmu
    grad_mu = max(min(grad_mu, 3.0), -3.0)
    mu -= lr * grad_mu
    mu = mu.clamp(0.0, 20.0)

    history.append((step, mu.item(), gamma_fitted))
    if step % 5 == 0:
        print(f'Step {step:2d}: mu={mu.item():.3f}, gamma_fit={gamma_fitted:.3f}')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].plot([h[0] for h in history], [h[1] for h in history], 'o-')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('mu'); axes[0].grid(alpha=0.3)
axes[0].set_title('Optimization of mu')

axes[1].plot([h[0] for h in history], [h[2] for h in history], 'o-', label='gamma_fit')
axes[1].axhline(target_gamma, color='r', ls='--', alpha=0.5, label=f'target={target_gamma}')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('gamma_fit'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('gamma_fit toward target')

axes[2].plot([h[0] for h in history], [(h[2] - target_gamma)**2 for h in history], 'o-')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Loss'); axes[2].grid(alpha=0.3)
axes[2].set_title('(gamma_fit - target)^2')
plt.tight_layout(); plt.show()

print(f'Final: mu={mu.item():.3f}, gamma_fit={history[-1][2]:.3f}, target={target_gamma}')

## Summary

1. **Pipeline:** N(mu, 1) -> square -> Voigt fit -> gamma_fit -> loss -> optimize mu
2. **Implicit differentiation** gives us dgamma/dmu without backpropagating through L-BFGS
3. The optimization converges: mu starts at 3.0 and finds the value that makes gamma_fit = 15.0
4. **Practical tricks:** log_gamma for positivity, gradient clipping for stability